In [ ]:
!pip install torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import VOCDetection
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 20
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3
IMG_SIZE = 224

VOC_CLASSES = [
    "aeroplane","bicycle","bird","boat","bottle",
    "bus","car","cat","chair","cow",
    "diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor"
]

CLASS2IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
class VOCMultiLabel(torch.utils.data.Dataset):
    def __init__(self, root, year="2012", image_set="train",
                 download=True, transform=None):
        self.base = VOCDetection(
            root=root,
            year=year,
            image_set=image_set,
            download=download
        )
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]

        if self.transform:
            img = self.transform(img)

        label = torch.zeros(NUM_CLASSES)

        objects = target["annotation"].get("object", [])

        if objects is None:
            objects = []
        elif isinstance(objects, dict):
            objects = [objects]

        for obj in objects:
            name = obj["name"]
            if name in CLASS2IDX:
                label[CLASS2IDX[name]] = 1.0

        return img, label

In [ ]:
full_dataset = VOCMultiLabel(
    root="./data",
    year="2012",
    image_set="train",
    download=True,
    transform=transform
)

print(len(full_dataset))

5717


In [ ]:
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0)

In [ ]:
for imgs, labels in train_loader:
    print(imgs.shape)
    print(labels.shape)
    break

torch.Size([16, 3, 224, 224])
torch.Size([16, 20])


In [ ]:
model = models.resnet18(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)

print("Model ready on:", DEVICE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Model ready on: cuda


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.5
)

In [ ]:
def compute_accuracy(logits, labels, threshold=0.5):
    preds = (torch.sigmoid(logits) >= threshold).float()
    correct = (preds == labels).all(dim=1).sum().item()
    return correct / labels.size(0)


def compute_ap(scores, labels):
    sorted_idx = np.argsort(-scores)
    labels_sorted = labels[sorted_idx]

    tp_cumsum = np.cumsum(labels_sorted)
    precision = tp_cumsum / (np.arange(len(labels_sorted)) + 1)
    recall = tp_cumsum / (labels_sorted.sum() + 1e-8)

    return np.trapz(precision, recall)


def compute_map(all_logits, all_labels):
    scores = 1 / (1 + np.exp(-all_logits))

    aps = []
    for c in range(NUM_CLASSES):
        if all_labels[:, c].sum() == 0:
            continue
        aps.append(compute_ap(scores[:, c], all_labels[:, c]))

    return np.mean(aps) if aps else 0.0

In [ ]:
imgs, labels = next(iter(train_loader))

imgs = imgs.to(DEVICE)
labels = labels.to(DEVICE)

logits = model(imgs)
loss = criterion(logits, labels)

print("Logits shape:", logits.shape)
print("Labels shape:", labels.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([16, 20])
Labels shape: torch.Size([16, 20])
Loss: 0.7892850041389465


In [ ]:
history = defaultdict(list)

for epoch in range(1, EPOCHS + 1):

    # =====================
    # TRAIN
    # =====================
    model.train()
    train_loss = 0.0
    train_acc = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_acc += (torch.sigmoid(logits) >= 0.5).float().eq(labels).all(dim=1).sum().item()

    train_loss /= len(train_ds)
    train_acc /= len(train_ds)

    # =====================
    # VALIDATION
    # =====================
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            loss = criterion(logits, labels)

            val_loss += loss.item() * imgs.size(0)
            val_acc += (torch.sigmoid(logits) >= 0.5).float().eq(labels).all(dim=1).sum().item()

            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_ds)
    val_acc /= len(val_ds)

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)

    val_map = compute_map(all_logits, all_labels)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_map"].append(val_map)

    print(f"Epoch [{epoch}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val mAP: {val_map:.4f}")

/tmp/ipykernel_10793/974145895.py:15: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(precision, recall)


Epoch [1/10] Train Loss: 0.1817 | Val Loss: 0.1168 | Val mAP: 0.7216
Epoch [2/10] Train Loss: 0.1222 | Val Loss: 0.1035 | Val mAP: 0.7635
Epoch [3/10] Train Loss: 0.1118 | Val Loss: 0.0981 | Val mAP: 0.7788
Epoch [4/10] Train Loss: 0.1038 | Val Loss: 0.0952 | Val mAP: 0.7841
Epoch [5/10] Train Loss: 0.1032 | Val Loss: 0.0958 | Val mAP: 0.7866
Epoch [6/10] Train Loss: 0.1000 | Val Loss: 0.0939 | Val mAP: 0.7890
Epoch [7/10] Train Loss: 0.0981 | Val Loss: 0.0925 | Val mAP: 0.7888
Epoch [8/10] Train Loss: 0.0975 | Val Loss: 0.0944 | Val mAP: 0.7907
Epoch [9/10] Train Loss: 0.0969 | Val Loss: 0.0936 | Val mAP: 0.7897
Epoch [10/10] Train Loss: 0.0954 | Val Loss: 0.0936 | Val mAP: 0.7918
